# 📝 Text Summarizer using T5 Transformer

This notebook fine-tunes a **T5-small** model on the **SAMSum dataset** (dialogue summaries).

**Pipeline:**
```
Load Data → Clean → Tokenize → Fine-tune T5 → Save Model → Summarize new dialogue
```

In [ ]:
# ============================================================
# CELL 1: Import Libraries
# ============================================================
# pandas       → load and manipulate CSV data
# T5Tokenizer  → converts raw text into token IDs (numbers) T5 understands
# T5ForConditionalGeneration → the T5 model architecture for text generation tasks
# Trainer      → HuggingFace training loop handler (handles backprop, eval, saving)
# TrainingArguments → config object for all training hyperparameters
# re           → regex library for text cleaning
# ============================================================

import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import re


In [ ]:
# ============================================================
# CELL 2: Load Dataset
# ============================================================
# SAMSum dataset: contains real-life messenger dialogues + human-written summaries
# train_data → 14,732 dialogue-summary pairs used to teach the model
# val_data   → 818 pairs used to evaluate model during training (not used for learning)
# ============================================================

import pandas as pd

train_data = pd.read_csv("/content/samsum-train.csv")
val_data = pd.read_csv("/content/samsum-validation.csv")


In [ ]:
# ============================================================
# CELL 3: Explore Data
# ============================================================
# .head()  → shows first 5 rows to understand the data structure
# .shape   → shows (rows, columns) — confirms how much data we have
# ============================================================

train_data.head()          # preview first 5 training samples


In [ ]:
print("Training set shape:", train_data.shape)    # e.g. (14732, 3)
print("Validation set shape:", val_data.shape)    # e.g. (818, 3)


In [ ]:
# ============================================================
# CELL 4: Random Sampling (reduce dataset size for faster training)
# ============================================================
# Full dataset = 14,732 rows → takes too long on free Colab GPU
# We sample 5000 train + 500 val rows → enough to fine-tune well
# random_state=42 → ensures same samples every run (reproducibility)
# reset_index(drop=True) → resets row numbers after sampling
# ============================================================

train_data = train_data.sample(n=5000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

print("Sampled training set shape:", train_data.shape)    # (5000, 3)
print("Sampled validation set shape:", val_data.shape)    # (500, 3)


In [ ]:
# ============================================================
# CELL 5: Data Cleaning Function
# ============================================================
# Raw dialogues contain noisy text: extra newlines, HTML tags, irregular spaces
# We clean all text before feeding it to the tokenizer
#
# re.sub(r"\r\n", " ", text)  → replaces Windows-style newlines with space
# re.sub(r"\s+", " ", text)    → collapses multiple spaces/tabs into single space
#                                  NOTE: \s+ (with backslash) is correct!
#                                  s+ (without backslash) would remove all 's' letters!
# re.sub(r"<.*?>", " ", text)   → removes HTML tags like <b>, <br>, </p> etc.
# .strip().lower()              → removes leading/trailing spaces, converts to lowercase
# ============================================================

import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)   # remove Windows newlines
    text = re.sub(r"\s+", " ", text)     # collapse multiple spaces (IMPORTANT: use \s not s)
    text = re.sub(r"<.*?>", " ", text)    # remove any HTML tags
    text = text.strip().lower()           # trim edges and lowercase everything
    return text


In [ ]:
# ============================================================
# CELL 6: Apply Cleaning to All Data
# ============================================================
# .apply(clean_data) → runs clean_data() on every row of that column
# We clean both 'dialogue' (input) and 'summary' (target/label)
# for both train and validation datasets
# ============================================================

train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary']  = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary']  = val_data['summary'].apply(clean_data)

print("Cleaning done!")
print("Sample cleaned dialogue:", train_data['dialogue'][0][:100])
print("Sample cleaned summary :", train_data['summary'][0])


In [ ]:
# ============================================================
# CELL 7: Load Tokenizer
# ============================================================
# T5Tokenizer converts raw text → token IDs (numbers) the model understands
# 'from_pretrained('t5-small')' loads the vocabulary T5 was originally trained with
# This ensures text is split into the exact same subwords T5 expects
# We reuse this same tokenizer at inference time (summarization)
# ============================================================

tokanizer = T5Tokenizer.from_pretrained('t5-small')

print("Tokenizer vocabulary size:", tokanizer.vocab_size)


In [ ]:
# ============================================================
# CELL 8: Tokenization Function
# ============================================================
# Converts each dialogue-summary pair into model-ready numeric tensors
#
# inputs = tokenizer(dialogue):
#   → 'input_ids'      : list of token IDs for the dialogue (the input to model)
#   → 'attention_mask' : 1 for real tokens, 0 for padding tokens
#
# target = tokenizer(summary):
#   → 'input_ids'      : token IDs for the expected summary (what model should output)
#
# inputs['labels'] = target['input_ids']:
#   → adds the summary token IDs as 'labels' so Trainer knows what output to expect
#   → during training, model tries to predict these labels and loss is computed
#
# padding='max_length' → pads shorter sequences to fixed length (all same size)
# truncation=True      → cuts sequences longer than max_length
# max_length=512       → max input (dialogue) length
# max_length=150       → max output (summary) length
# ============================================================

def tokanize(data):
    max_length = 512
    inputs = tokanizer(
        data['dialogue'],
        padding='max_length',
        max_length=max_length,
        truncation=True
    )
    target = tokanizer(
        data['summary'],
        padding='max_length',
        max_length=150,
        truncation=True
    )
    inputs['labels'] = target['input_ids']   # attach summary token IDs as training labels
    return inputs


In [ ]:
# ============================================================
# CELL 9: Apply Tokenization to Entire Dataset
# ============================================================
# .apply(tokanize, axis=1) → applies tokanize() to each row (axis=1 = row-wise)
# .tolist() → converts result to a Python list of dicts
# Each element = {'input_ids': [...], 'attention_mask': [...], 'labels': [...]}
# Trainer expects this list format to feed batches during training
# ============================================================

train_dataset = train_data.apply(tokanize, axis=1).tolist()
val_dataset   = val_data.apply(tokanize, axis=1).tolist()

print("Total training samples:", len(train_dataset))
print("Total validation samples:", len(val_dataset))
print("Keys in each sample:", list(train_dataset[0].keys()))
print("Input length:", len(train_dataset[0]['input_ids']))    # should be 512
print("Label length:", len(train_dataset[0]['labels']))       # should be 150


In [ ]:
# ============================================================
# CELL 10: Load Pretrained T5-Small Model
# ============================================================
# T5ForConditionalGeneration = T5 model designed for sequence-to-sequence tasks
# (text-in → text-out), perfect for summarization, translation, Q&A
#
# 't5-small' = smallest T5 variant (~60M parameters)
#   → fast to fine-tune on free Colab GPU
#   → already knows English grammar and language patterns
#   → we just teach it the summarization task via fine-tuning
# ============================================================

model = T5ForConditionalGeneration.from_pretrained('t5-small')

print("Model loaded!")
print("Model parameters:", sum(p.numel() for p in model.parameters()), "parameters")


In [ ]:
# ============================================================
# CELL 11: Device Setup (GPU / MPS / CPU)
# ============================================================
# Training on GPU is ~75x faster than CPU
# This code auto-detects the best available hardware:
#   → 'cuda'  : NVIDIA GPU (Google Colab T4) ← fastest
#   → 'mps'   : Apple Silicon GPU (M1/M2 Mac)
#   → 'cpu'   : fallback if no GPU available ← very slow
#
# torch.backends.mps.is_available() → checks Apple Silicon GPU
# torch.cuda.is_available()         → checks NVIDIA GPU
# ============================================================

import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


In [ ]:
# ============================================================
# CELL 12: Training Arguments (Hyperparameters)
# ============================================================
# TrainingArguments = config object that controls the entire training process
#
# output_dir='./results'           → folder to save model checkpoints after each epoch
# num_train_epochs=6               → full passes over the training data (6 loops)
# per_device_train_batch_size=8    → 8 samples processed at once during training
# per_device_eval_batch_size=8     → 8 samples processed at once during evaluation
# eval_strategy='epoch'            → run validation after each epoch to track progress
# save_strategy='epoch'            → save a checkpoint after each epoch
# warmup_steps=500                 → gradually increase learning rate for first 500 steps
#                                     (prevents large weight updates at start = more stable)
# weight_decay=0.01                → L2 regularization to prevent overfitting
#                                     (penalizes large weights, forces model to generalize)
# ============================================================

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    warmup_steps=500
)


In [ ]:
# ============================================================
# CELL 13: Initialize Trainer
# ============================================================
# Trainer = HuggingFace's built-in training loop manager
# It handles everything automatically:
#   → feeding batches to model
#   → computing loss (how wrong the model is)
#   → backpropagation (adjusting weights)
#   → evaluation on val_dataset after each epoch
#   → saving checkpoints to output_dir
#
# model         → the T5-small model to fine-tune
# args          → training config from TrainingArguments above
# train_dataset → tokenized training samples
# eval_dataset  → tokenized validation samples (used only to monitor, not to train)
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


In [ ]:
# ============================================================
# CELL 14: Train the Model
# ============================================================
# trainer.train() starts the full fine-tuning loop:
#   1. Feeds batches of (dialogue tokens → summary labels)
#   2. Model predicts summary token IDs
#   3. Cross-entropy loss computed between prediction and true labels
#   4. Gradients calculated via backpropagation
#   5. Optimizer (AdamW) updates model weights to reduce loss
#   6. Repeats for all batches × 6 epochs
#
# NOTE: We assign result to a variable but actual trained model
# is inside trainer.model (not this variable)
# Expected time: ~20-30 min on T4 GPU
# ============================================================

# Training Model
trainer.train()


In [ ]:
# ============================================================
# CELL 15: Save Fine-Tuned Model
# ============================================================
# After training, we save model weights + tokenizer to disk
# so we don't need to retrain every time
#
# trainer.model.save_pretrained('./saved_summary_model')
#   → saves model weights (pytorch_model.bin) and config.json
#
# tokanizer.save_pretrained('./saved_summary_model')
#   → saves tokenizer vocab and config files
#
# Later we can reload with:
#   model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
# ============================================================

trainer.model.save_pretrained('./saved_summary_model')
tokanizer.save_pretrained('./saved_summary_model')

print("Model saved to ./saved_summary_model")


In [ ]:
# ============================================================
# CELL 16: Reload Saved Model for Inference
# ============================================================
# Load fine-tuned model weights from disk
# Load tokenizer from original 't5-small' (NOT from saved_summary_model)
#   → reason: tokenizer saved inside the model folder may have encoding issues
#   → original t5-small tokenizer is always clean and reliable
# ============================================================

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load fine-tuned model weights
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')

# Load tokenizer from original t5-small (safer than loading from saved folder)
tokenizer = T5Tokenizer.from_pretrained('t5-small')

print("Model and tokenizer loaded!")


In [ ]:
# ============================================================
# CELL 17: Summarization Function
# ============================================================
# Full pipeline: raw dialogue text → clean → tokenize → generate → decode → summary
#
# Step 1: clean_data(dialogue)
#   → removes noise (extra spaces, newlines, HTML tags)
#
# Step 2: tokenizer(dialogue, ...)
#   → converts cleaned text to token IDs
#   → .to(device) moves tensors to GPU for fast inference
#
# Step 3: model.generate(...)
#   → generates summary token IDs using beam search
#   → input_ids       : tokenized dialogue
#   → attention_mask  : tells model which tokens are real vs padding
#   → max_length=200  : summary won't exceed 200 tokens
#   → min_length=60   : forces a reasonably detailed summary
#   → num_beams=4     : beam search — explores 4 paths, keeps best
#   → length_penalty=2.0 : values >1.0 encourage longer summaries
#   → early_stopping  : stops beam search when all beams hit end token
#
# Step 4: tokenizer.decode(...)
#   → converts output token IDs back to human-readable text
#   → skip_special_tokens=True → removes <pad>, </s> etc.
# ============================================================

def summarize_dialogue(dialogue):
    # Step 1: Clean the input text
    dialogue = clean_data(dialogue)

    # Step 2: Tokenize — convert text to token IDs
    inputs = tokenizer(
        dialogue,
        max_length=512,           # max input length
        padding='max_length',     # pad to fixed length
        truncation=True,          # cut if longer than 512
        return_tensors='pt'       # return PyTorch tensors
    ).to(device)                  # move to GPU

    # Step 3: Generate summary token IDs
    model.to(device)              # ensure model is on GPU
    target = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=200,           # max summary length
        min_length=60,            # min summary length
        num_beams=4,              # beam search width
        length_penalty=2.0,       # encourage longer output
        early_stopping=True       # stop when best beam is complete
    )

    # Step 4: Decode token IDs back to readable text
    summary = tokenizer.decode(
        target[0],
        skip_special_tokens=True,           # remove special tokens
        clean_up_tokenization_spaces=True   # fix spacing artifacts
    )

    return summary


In [ ]:
# ============================================================
# CELL 18: Test the Model with Sample Dialogue
# ============================================================
# We provide a multi-turn dialogue between a Reporter and an Expert
# The model should summarize the key points into a short paragraph
# ============================================================

test_dialouge = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance.
Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experience.
Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks.
Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on.
Reporter: Governments and organizations are beginning to introduce regulations to guide the development of AI.
Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, are difficult to interpret.
Reporter: Experts also highlight the importance of responsible AI development, including data privacy.
Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential.
"""

# Run summarization
summary = summarize_dialogue(test_dialouge)

print("=" * 60)
print("GENERATED SUMMARY:")
print("=" * 60)
print(summary)
